# Stage 02 — Data Preparation

Load raw data, merge datasets, construct variables, write `data/dataset.parquet`.
Follow `skills/stage_02.md` for detailed instructions.

In [1]:
from paths import *
import json
import numpy as np
import pandas as pd
import pyreadstat

spec = json.loads(PAPER_SPEC.read_text())
print('Identification type:', spec['identification']['type'])
print('Primary file:', spec['identification']['primary_data_file'])
print('Secondary datasets:', spec['identification'].get('secondary_datasets', []))

Identification type: IV
Primary file: country.dta
Secondary datasets: ['ethnic.dta', 'ethnicpair.dta']


In [2]:
# --- AGENT: load primary dataset ---
primary_file = RAW_DATA_DIR / spec['identification']['primary_data_file']
df, meta = pyreadstat.read_dta(str(primary_file))
print(f'Primary dataset shape: {df.shape}')
df.head()

Primary dataset shape: (208, 179)


,id,code,country,pop1,pop1000,pop1500,pd1,pd1000,pd1500,ur1500,...,cleanlimfst,cleanpd1500fst,cleancompfst,cleangdptrust,cleangdparticles,conley_coord1,conley_coord2,conley_cutoff1,conley_cutoff2,conley_const
0,1.0,ABW,Aruba,7.979784e+01,1.595957e+02,2.393935e+02,0.443321,0.886643,1.329964,NaN,...,0,0,0,0,0,NaN,NaN,5,5,1
1,2.0,ADO,Andorra,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,0,0,0,0,NaN,NaN,5,5,1
2,3.0,AFG,Afghanistan,2.000000e+06,2.250000e+06,2.000000e+06,3.067061,3.450444,3.067061,NaN,...,0,1,1,0,1,4.699989,-0.508255,5,5,1
3,4.0,AGO,Angola,2.383606e+05,9.534425e+05,1.906885e+06,0.191193,0.764773,1.529546,NaN,...,0,1,1,0,0,-0.320201,4.631516,5,5,1
4,5.0,ALB,Albania,2.000000e+05,2.000000e+05,2.000000e+05,7.299270,7.299270,7.299270,2.5,...,0,1,1,1,1,0.407309,0.092272,5,5,1


In [3]:
# --- AGENT: merge secondary datasets if needed ---
# ethnic.dta and ethnicpair.dta contain ethnicity-level data used in robustness
# checks (e.g., within-country diversity variation). They are NOT used in the
# main country-level analysis (Tables 1-7 in the paper). We skip merging them
# here; the country-level dataset is self-contained.
# If ethnic-level robustness checks are needed, load them in a separate notebook.

print("Secondary datasets (ethnic.dta, ethnicpair.dta) are ethnicity-level data.")
print("They are NOT merged here — the main analysis uses country-level observations only.")
print(f"Country-level dataset shape after primary load: {df.shape}")

Secondary datasets (ethnic.dta, ethnicpair.dta) are ethnicity-level data.
They are NOT merged here — the main analysis uses country-level observations only.
Country-level dataset shape after primary load: (208, 179)


In [4]:
# --- AGENT: construct variables ---
# All variables required for replication and DML are checked below.
# Because country.dta already contains log-transformed and squared terms,
# most construction is verification rather than computation.

# --- Outcome variable ---
# ln_gdppc2000 already exists as log(income per capita in 2000 CE).
# No transformation needed.
assert 'ln_gdppc2000' in df.columns, "ln_gdppc2000 missing from data"
print(f"[OK] Outcome 'ln_gdppc2000' already present. Non-null: {df['ln_gdppc2000'].notna().sum()}")

# --- Treatment variables (quadratic functional form) ---
# pdiv_aa: ancestry-adjusted predicted genetic diversity (linear term)
# pdiv_aa_sqr: its square — already present in the raw data
assert 'pdiv_aa' in df.columns, "pdiv_aa missing from data"
assert 'pdiv_aa_sqr' in df.columns, "pdiv_aa_sqr missing from data"
print(f"[OK] Treatment 'pdiv_aa' already present. Non-null: {df['pdiv_aa'].notna().sum()}")
print(f"[OK] Squared treatment 'pdiv_aa_sqr' already present. Non-null: {df['pdiv_aa_sqr'].notna().sum()}")

# Cross-check: verify that pdiv_aa_sqr == pdiv_aa ** 2 (within floating-point tolerance)
mask = df['pdiv_aa'].notna() & df['pdiv_aa_sqr'].notna()
max_diff = (df.loc[mask, 'pdiv_aa_sqr'] - df.loc[mask, 'pdiv_aa'] ** 2).abs().max()
print(f"[CHECK] Max abs diff between pdiv_aa_sqr and pdiv_aa^2: {max_diff:.2e}")

# --- Instrument ---
assert 'mdist_addis' in df.columns, "mdist_addis missing from data"
print(f"[OK] Instrument 'mdist_addis' already present. Non-null: {df['mdist_addis'].notna().sum()}")

# --- Controls (all should already be log-transformed in the data) ---
controls = ['ln_yst', 'ln_arable', 'ln_abslat', 'ln_soilsuit', 'abslat', 'temp', 'precip']
for c in controls:
    assert c in df.columns, f"Control '{c}' missing from data"
    print(f"[OK] Control '{c}' present. Non-null: {df[c].notna().sum()}")

# --- Continent dummies (fixed effects) ---
# The data contains a 'continent' string column and individual dummy columns:
#   africa, europe, asia, oceania, americas
# Both are kept. We also create a clean categorical version for convenience.
assert 'continent' in df.columns, "'continent' column missing from data"
print(f"\n[OK] 'continent' column present. Value counts:")
print(df['continent'].value_counts(dropna=False).to_string())

# Verify the five binary continent dummies already exist
continent_dummies = ['africa', 'europe', 'asia', 'oceania', 'americas']
for d in continent_dummies:
    assert d in df.columns, f"Continent dummy '{d}' missing from data"
print(f"[OK] Continent dummies present: {continent_dummies}")

# --- Historical outcome (used in replication of Tables 1-3) ---
assert 'ln_pd1500' in df.columns, "ln_pd1500 missing from data"
print(f"\n[OK] Historical outcome 'ln_pd1500' present. Non-null: {df['ln_pd1500'].notna().sum()}")

print("\n[DONE] All required variables verified. No new columns constructed (all pre-computed in raw data).")
print(f"Dataset shape: {df.shape}")

[OK] Outcome 'ln_gdppc2000' already present. Non-null: 187
[OK] Treatment 'pdiv_aa' already present. Non-null: 164
[OK] Squared treatment 'pdiv_aa_sqr' already present. Non-null: 164
[CHECK] Max abs diff between pdiv_aa_sqr and pdiv_aa^2: 2.96e-08
[OK] Instrument 'mdist_addis' already present. Non-null: 207
[OK] Control 'ln_yst' present. Non-null: 164
[OK] Control 'ln_arable' present. Non-null: 196
[OK] Control 'ln_abslat' present. Non-null: 205
[OK] Control 'ln_soilsuit' present. Non-null: 164
[OK] Control 'abslat' present. Non-null: 205
[OK] Control 'temp' present. Non-null: 184
[OK] Control 'precip' present. Non-null: 184

[OK] 'continent' column present. Value counts:
continent
Africa      54
Asia        49
Europe      46
Americas    42
Oceania     17
[OK] Continent dummies present: ['africa', 'europe', 'asia', 'oceania', 'americas']

[OK] Historical outcome 'ln_pd1500' present. Non-null: 184

[DONE] All required variables verified. No new columns constructed (all pre-computed in r

In [5]:
# --- Validate ---
key_vars = ([spec['identification']['outcome_variable'],
             spec['identification']['treatment_variable']] +
            ([spec['identification']['instrument']]
             if 'instrument' in spec['identification'] else []))

print('Missingness in key variables:')
print(df[key_vars].isnull().sum())
print(f'\nTotal rows with all key vars present: {df[key_vars].dropna().shape[0]}')

assert df[key_vars].dropna().shape[0] >= 30, 'Too few complete observations'
print('\nDescriptive statistics:')
df[key_vars].describe()

Missingness in key variables:
ln_gdppc2000    21
pdiv_aa         44
mdist_addis      1
dtype: int64

Total rows with all key vars present: 162

Descriptive statistics:


,ln_gdppc2000,pdiv_aa,mdist_addis
count,187.000000,164.000000,207.000000
mean,8.520773,0.726763,9.731676
std,1.176689,0.026937,7.381739
min,5.883732,0.627887,0.000000
25%,7.621068,0.721518,4.352502
50%,8.558548,0.733474,5.972080
75%,9.532299,0.742818,17.391890
max,10.783473,0.774301,26.770687


In [6]:
# --- Write output ---
df.to_parquet(str(DATASET_PATH), index=False)
print(f'✓ Written: {DATASET_PATH}')
print(f'  Shape: {df.shape}')
print(f'  Columns: {list(df.columns)}')

✓ Written: C:\Users\qgallea\Dropbox\work\claude_code\causal_ml_extension\papers\01_out_of_africa\data\dataset.parquet
  Shape: (208, 179)
  Columns: ['id', 'code', 'country', 'pop1', 'pop1000', 'pop1500', 'pd1', 'pd1000', 'pd1500', 'ur1500', 'gdppc2000', 'trust', 'articles', 'adiv', 'adiv_sqr', 'pdiv', 'pdiv_sqr', 'pdiv_aa', 'pdiv_aa_sqr', 'pdivhmi', 'pdivhmi_sqr', 'pdivhmi_aa', 'pdivhmi_aa_sqr', 'ahomo', 'ahomo_sqr', 'phomo', 'phomo_sqr', 'phomo_aa', 'phomo_aa_sqr', 'mdist_hgdp', 'mdist_hgdp_sqr', 'mdist_addis', 'mdist_addis_sqr', 'mdist_addis_aa', 'mdist_addis_aa_sqr', 'mdist_london', 'mdist_london_sqr', 'mdist_tokyo', 'mdist_tokyo_sqr', 'mdist_mexico', 'mdist_mexico_sqr', 'aerial', 'aerial_sqr', 'aerial_aa', 'aerial_aa_sqr', 'frontdist1', 'frontdist1000', 'frontdist1500', 'hmicost_hgdp', 'hmicost_hgdp_sqr', 'hmicost_addis', 'hmicost_addis_sqr', 'hmicost_addis_aa', 'hmicost_addis_aa_sqr', 'fstdist1500_uk', 'fstdist1500_eth', 'fstdistwtd_usa', 'fstdistwtd_eth', 'yst', 'yst_aa', 'yst15